# Telugu Unicode WordPiece Tokenizer Training

## What is WordPiece?

**WordPiece** is a subword tokenization algorithm that handles rare words better than BPE.

### BPE vs WordPiece

| Aspect | BPE | WordPiece |
|--------|-----|----------|
| **Learning** | Merges frequent pairs | Uses likelihood scoring |
| **Rare words** | Can split inefficiently | Breaks down intelligently |
| **UNK rate** | Higher for rare words | Lower (better coverage) |
| **Use case** | General language | Better for low-resource languages |

### Example: Rare Telugu word
```
Word: "నిర్ణయించు" (decide - rare)

BPE:       ['నిర్', '##ణయ', '##ించు']     (may not learn this pattern)
WordPiece: ['నిర్', '##ణయ', '##ించు']     (learns likelihood of each subword)
```

**Advantages:**
- ✅ Better OOV (out-of-vocabulary) handling
- ✅ Lower UNK rate
- ✅ Smarter rare word splitting
- ✅ Better for Telugu (low-resource language)
- ✅ Probability-based (more linguistic)


## ⚠️ Important: Trained from Scratch (No Pretrained Tokenizers)

**This notebook trains a WordPiece tokenizer from scratch on the Telugu corpus.**

- Uses the standalone `tokenizers` library (NOT `transformers`)
- Starts with a blank `models.WordPiece()` with no pretrained vocabulary
- **Unicode-level alphabet** (Unicode character set, not bytes)
- **No `.from_pretrained()` call anywhere** — all vocab learned purely from corpus
- Satisfies project constraint: *No pretrained models*


In [ ]:
import json
import logging
import random
import string
import gc
import os
import tempfile
from datetime import datetime
from pathlib import Path
from typing import Optional

from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, decoders, processors, trainers

# ============================================================================
# 🔧 SET SEED FOR REPRODUCIBILITY
# ============================================================================
SEED = 42
random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
logger_init = logging.getLogger()
print(f"✓ Seed set to {SEED} for reproducibility")

# Try to import psutil for memory monitoring (optional)
try:
    import psutil
    HAS_PSUTIL = True
except ImportError:
    HAS_PSUTIL = False
    print("⚠️  psutil not available - memory monitoring disabled")

# ============================================================================
# ⚙️ CONFIGURATION: DATA ROOT PATH
# ============================================================================
DATA_ROOT = Path("/kaggle/input/datasets/kspsvlnsiddardha/lma-slm/telugui/data")

TRAIN_DIR = DATA_ROOT / "train"
VAL_DIR = DATA_ROOT / "val"
TEST_DIR = DATA_ROOT / "test"
TOKENIZER_DIR = Path("/kaggle/working/")
TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)

print(f"✓ Data root: {DATA_ROOT}")
print(f"✓ Tokenizer dir: {TOKENIZER_DIR}")
print(f"✓ Train dir exists: {TRAIN_DIR.exists()}")
print(f"✓ Val dir exists: {VAL_DIR.exists()}")
print(f"✓ Test dir exists: {TEST_DIR.exists()}")

# ============================================================================
# Language & tokenizer config
# ============================================================================
LANG = "Telugu"
LANG_SHORT = "telugu"
SPECIAL_TOKENS = ["[PAD]", "[UNK]", "[BOS]", "[EOS]"]

# Logging setup
logging.basicConfig(
    level=logging.INFO,
    format="[%(levelname)s] %(message)s"
)
logger = logging.getLogger(__name__)

In [ ]:
# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

def estimate_corpus_tokens(total_bytes: int, bytes_per_token: float = 4.0) -> int:
    """Rough token estimate from corpus size."""
    return int(total_bytes / bytes_per_token)

def compute_vocab_size(total_tokens_estimate: int) -> int:
    """Tiered vocab size heuristic."""
    if total_tokens_estimate < 50_000_000:
        return 8_000
    elif total_tokens_estimate < 200_000_000:
        return 16_000
    elif total_tokens_estimate < 1_000_000_000:
        return 32_000
    else:
        return 50_000

def gather_training_files(split_dirs: list) -> list:
    """Find all *.txt files in split directories."""
    files = []
    for split_dir in split_dirs:
        if split_dir.exists():
            files.extend(sorted(split_dir.glob("*.txt")))
    return files

def total_bytes(files: list) -> int:
    """Compute total size of files."""
    return sum(f.stat().st_size for f in files if f.exists())

# ---- Tokenizer Construction (UNICODE-LEVEL WORDPIECE) ----
def create_unicode_wordpiece_tokenizer() -> Tokenizer:
    """Create a Unicode-level WordPiece tokenizer."""
    tokenizer = Tokenizer(models.WordPiece(unk_token="[UNK]"))
    tokenizer.normalizer = normalizers.NFC()
    tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()  # Split on whitespace
    tokenizer.decoder = decoders.WordPiece(prefix="##")
    return tokenizer

def build_wordpiece_trainer(vocab_size: int) -> trainers.WordPieceTrainer:
    """Build a Unicode-level WordPiece trainer."""
    # Unicode alphabet: all printable ASCII + Telugu range
    unicode_alphabet = list(string.printable)
    # Add Telugu script range (U+0C00 to U+0C7F)
    for i in range(0x0C00, 0x0C80):
        unicode_alphabet.append(chr(i))

    return trainers.WordPieceTrainer(
        vocab_size=vocab_size,
        min_frequency=2,
        special_tokens=SPECIAL_TOKENS,
        initial_alphabet=unicode_alphabet,
        show_progress=True,
    )

print("✅ ALL FUNCTIONS DEFINED - Ready to use!")

In [ ]:
# ============================================================================
# Discover corpus files
# ============================================================================

logger.info("Discovering corpus files...")
train_files = gather_training_files([TRAIN_DIR])
val_files = gather_training_files([VAL_DIR])
test_files = gather_training_files([TEST_DIR])

logger.info(f"Train files: {[f.name for f in train_files]}")
logger.info(f"Val files: {[f.name for f in val_files]}")
logger.info(f"Test files: {[f.name for f in test_files]}")

# Compute vocab size from train+val
train_val_files = train_files + val_files
train_val_bytes = total_bytes(train_val_files)
train_val_tokens = estimate_corpus_tokens(train_val_bytes)
vocab_size = compute_vocab_size(train_val_tokens)

print(f"\n📊 Corpus stats (train+val):")
print(f"  Total bytes: {train_val_bytes / (1024**3):.2f} GB")
print(f"  Estimated tokens: {train_val_tokens:,}")
print(f"  Vocab size (heuristic): {vocab_size:,}")

In [ ]:
# ============================================================================
# Memory cleanup before training
# ============================================================================

if HAS_PSUTIL:
    process = psutil.Process(os.getpid())
    mem_before = process.memory_info().rss / (1024**3)
    logger.info(f"Memory usage before cleanup: {mem_before:.2f} GB")

gc.collect()

if HAS_PSUTIL:
    mem_after = process.memory_info().rss / (1024**3)
    logger.info(f"Memory usage after cleanup: {mem_after:.2f} GB")

In [ ]:
# ============================================================================
# Train tokenizer (UNICODE-LEVEL WORDPIECE) - BATCHED APPROACH
# ============================================================================

logger.info("Creating UNICODE-LEVEL WordPiece tokenizer...")
tokenizer = create_unicode_wordpiece_tokenizer()

logger.info("Building Unicode WordPiece trainer...")
trainer = build_wordpiece_trainer(vocab_size)

logger.info("Training Unicode WordPiece with BATCHED APPROACH (memory-efficient)...")
logger.info("Processing large files in chunks to prevent kernel crash...")

# Create temporary directory for batches
temp_dir = Path(tempfile.gettempdir()) / "telugu_wp_batches"
temp_dir.mkdir(exist_ok=True)

def create_file_batches(filepath, batch_size_mb=500):
    """Split a large file into smaller batches (in MB)."""
    batch_size_bytes = batch_size_mb * (1024**2)
    batch_files = []
    batch_num = 0
    
    current_batch = []
    current_size = 0
    
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line_bytes = len(line.encode('utf-8'))
            current_batch.append(line)
            current_size += line_bytes
            
            if current_size >= batch_size_bytes:
                # Save batch
                batch_file = temp_dir / f"{filepath.stem}_batch_{batch_num}.txt"
                with open(batch_file, 'w', encoding='utf-8') as bf:
                    bf.writelines(current_batch)
                batch_files.append(batch_file)
                batch_num += 1
                current_batch = []
                current_size = 0
        
        # Save remaining lines
        if current_batch:
            batch_file = temp_dir / f"{filepath.stem}_batch_{batch_num}.txt"
            with open(batch_file, 'w', encoding='utf-8') as bf:
                bf.writelines(current_batch)
            batch_files.append(batch_file)
    
    return batch_files

# Collect all batches from train+val
all_batches = []
for file_path in train_val_files:
    logger.info(f"Creating batches from {file_path.name}...")
    batches = create_file_batches(file_path, batch_size_mb=300)  # 300MB per batch
    all_batches.extend(batches)
    logger.info(f"  Created {len(batches)} batches from {file_path.name}")

logger.info(f"Total batches to process: {len(all_batches)}")

# Train on batches one by one
try:
    for i, batch_file in enumerate(all_batches, 1):
        logger.info(f"Training on batch {i}/{len(all_batches)}: {batch_file.name}")
        
        # Clear memory before each batch
        gc.collect()
        
        # Train on single batch
        tokenizer.train(
            files=[str(batch_file)],
            trainer=trainer,
        )
        
        # Show progress
        vocab_size_so_far = tokenizer.get_vocab_size()
        logger.info(f"  Batch {i} complete. Vocab size: {vocab_size_so_far:,}")
    
    logger.info("✓ Training complete on all batches!")

except Exception as e:
    logger.error(f"Error during batch training: {e}")
    logger.info("Using best available tokenizer state...")

finally:
    # Cleanup temp files
    logger.info("Cleaning up temporary batch files...")
    for batch_file in all_batches:
        try:
            batch_file.unlink()
        except:
            pass

print("✓ Batch training complete")

In [ ]:
# ============================================================================
# Save tokenizer and config
# ============================================================================

tokenizer_path = TOKENIZER_DIR / f"{LANG_SHORT}_wp_tokenizer.json"
logger.info(f"Saving tokenizer to {tokenizer_path.name}...")
tokenizer.save(str(tokenizer_path))

vocab_actual = tokenizer.get_vocab_size()
logger.info(f"Vocab size actual: {vocab_actual:,}")
if vocab_actual != vocab_size:
    logger.warning(f"  Note: actual ({vocab_actual}) differs from requested ({vocab_size})")

# Save config
config = {
    "language": LANG,
    "tokenizer_type": "WordPiece",
    "model": "Unicode WordPiece",
    "normalizer": "NFC",
    "pre_tokenizer": "Whitespace",
    "decoder": "WordPiece (with ## prefix for subwords)",
    "vocab_size_requested": vocab_size,
    "vocab_size_actual": vocab_actual,
    "special_tokens": SPECIAL_TOKENS,
    "created_at": datetime.now().isoformat(),
}

config_path = TOKENIZER_DIR / "wp_tokenizer_config.json"
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)
logger.info(f"Saved config to {config_path.name}")

print(f"✓ Tokenizer and config saved")

In [ ]:
# ============================================================================
# Quick evaluation and statistics
# ============================================================================

from collections import Counter

logger.info("Evaluating WordPiece tokenizer on test set...")

# Evaluate on sample of test set
token_freq = Counter()
total_tokens = 0
total_lines = 0
unk_count = 0
roundtrip_count = 0

for test_file in test_files:
    if not test_file.exists():
        continue
    with open(test_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line.strip():
                continue
            total_lines += 1
            
            # Encode and decode
            encoded = tokenizer.encode(line)
            decoded = tokenizer.decode(encoded.ids)
            
            total_tokens += len(encoded.ids)
            token_freq.update(encoded.ids)
            
            # Count UNK tokens
            unk_id = tokenizer.token_to_id("[UNK]")
            unk_count += sum(1 for tid in encoded.ids if tid == unk_id)
            
            # Count roundtrip matches (space handling may differ)
            if decoded.replace(" ", "") == line.replace(" ", ""):
                roundtrip_count += 1

# Compute stats
vocab_coverage = len(token_freq) / vocab_actual * 100
unk_rate = 100.0 * unk_count / total_tokens if total_tokens > 0 else 0
avg_tokens_per_line = total_tokens / total_lines if total_lines > 0 else 0
roundtrip_rate = 100.0 * roundtrip_count / total_lines if total_lines > 0 else 0

print(f"\n📊 WORDPIECE TOKENIZER REPORT:")
print(f"  Vocab size: {vocab_actual:,}")
print(f"  Vocab coverage (used in test): {vocab_coverage:.2f}%")
print(f"  Test lines: {total_lines:,}")
print(f"  Total test tokens: {total_tokens:,}")
print(f"  Avg tokens/line: {avg_tokens_per_line:.2f}")
print(f"  UNK rate: {unk_rate:.4f}%")
print(f"  Roundtrip match: {roundtrip_rate:.1f}%")

print(f"\n📈 Top-20 Most Frequent Tokens:")
for i, (token_id, count) in enumerate(token_freq.most_common(20), 1):
    token_str = tokenizer.decode([token_id])
    token_display = repr(token_str) if token_str in [' ', '\n', '\t'] else token_str
    percent = 100.0 * count / total_tokens
    print(f"  {i:2d}. {token_display:25s} id={token_id:5d} count={count:8d} ({percent:.2f}%)")

In [ ]:
# ============================================================================
# Test cases: Rare words and OOV handling (WordPiece advantage)
# ============================================================================

print("\n" + "="*70)
print("TEST CASES: RARE WORD HANDLING (WordPiece Advantage)")
print("="*70)

# Test 1: Common words
print("\n🔍 Common Words (should be single tokens):")
common_words = [
    "ఇది",
    "చదువు",
    "భారత",
]

for word in common_words:
    encoded = tokenizer.encode(word)
    tokens = [tokenizer.decode([tid]) for tid in encoded.ids]
    print(f"  '{word}' → {tokens} ({len(encoded.ids)} tokens)")

# Test 2: Rare/compound words (where WordPiece shines)
print("\n⭐ Rare/Compound Words (WordPiece handles better):")
rare_words = [
    "నిర్ణయించు",  # rare: decide
    "పరిశీలించండి",  # rare: examine
    "సంతృప్తిచెందారు",  # rare: satisfied
]

for word in rare_words:
    encoded = tokenizer.encode(word)
    tokens = [tokenizer.decode([tid]) for tid in encoded.ids]
    unk_id = tokenizer.token_to_id("[UNK]")
    has_unk = unk_id in encoded.ids
    unk_mark = " ⚠️ HAS UNK" if has_unk else " ✓ NO UNK"
    print(f"  '{word}' → {tokens} ({len(encoded.ids)} tokens){unk_mark}")

# Test 3: Sentence tokenization
print("\n📝 Full Sentences:")
sentences = [
    "ఇది ఒక పరీక్ష వాక్యం.",
    "తెలుగు భాష చాలా ందగా ఉంది.",
]

for sentence in sentences:
    encoded = tokenizer.encode(sentence)
    print(f"\n  Input: '{sentence}'")
    print(f"  Tokens: {len(encoded.ids)}")
    print(f"  Token IDs: {encoded.ids[:20]}..." if len(encoded.ids) > 20 else f"  Token IDs: {encoded.ids}")
    
    # Show individual tokens
    tokens = [tokenizer.decode([tid]) for tid in encoded.ids]
    print(f"  Decoded: {' '.join(tokens[:10])}..." if len(tokens) > 10 else f"  Decoded: {' '.join(tokens)}")

## Summary: WordPiece vs BPE

✅ **WordPiece advantages for Telugu:**
- **Better rare word handling** - Intelligently breaks down unknown words
- **Lower UNK rate** - Fewer completely unknown tokens
- **Subword units with ##** - Clearer subword boundaries (##ణయ = continuation)
- **Probability-based** - Uses likelihood instead of frequency alone
- **Better for low-resource** - Designed for languages with less training data

📊 **When to use which:**
- **BPE**: When you have very large corpus (>1B tokens)
- **WordPiece**: For Telugu (low-resource), better OOV handling

**Output files:**
- `telugu_wp_tokenizer.json` - Trained WordPiece tokenizer
- `wp_tokenizer_config.json` - Configuration and metadata


In [ ]:
# ============================================================================
# Test on sample Telugu text
# ============================================================================

print("\n" + "="*70)
print("TEST: Sample Telugu Text Tokenization")
print("="*70)

test_samples = [
    "ఇది ఒక పరీక్ష వాక్యం.",
    "తెలుగు ఎక్కువ భాషలలో మాట్లాడబడుతుంది.",
    "నేను ఒక విద్యార్థిని.",
]

for i, text in enumerate(test_samples, 1):
    print(f"\n{i}. Input: {text}")
    encoded = tokenizer.encode(text)
    print(f"   Token IDs: {encoded.ids}")
    
    # Decode individual tokens
    tokens = [tokenizer.decode([tid]) for tid in encoded.ids]
    print(f"   Tokens: {tokens}")
    print(f"   Total: {len(encoded.ids)} tokens")